In [ ]:
from transport.routes_generator import citygraph_dataset
from omegaconf import OmegaConf, DictConfig
from tqdm import tqdm
from pathlib import Path
from hydra import initialize_config_dir, compose

In [ ]:
CFG_DIR = "/root/transport/transport/routes_generator/cfg"

In [ ]:
dataset = citygraph_dataset.DynamicCityGraphDataset(
    min_nodes=25,
    max_nodes=25,
    edge_keep_prob=0.7,
    data_type=citygraph_dataset.MIXED,  # or any other type you want
    directed=False,
    fully_connected_demand=True,  # default SIDE_LENGTH_M
    mumford_style=True,
    pos_only=False
)

# Generate graphs
n_graphs = 1000
graphs = [dataset.generate_graph(draw=False) for _ in tqdm(range(n_graphs))]

In [ ]:
import pickle
from pathlib import Path

save_path = Path('./datasets/20_nodes/mixed')
if not save_path.exists():
    save_path.mkdir(parents=True)

with open(save_path / 'raw_graphs_1000.pkl', 'wb') as ff:
    pickle.dump(graphs, ff)

In [ ]:
from transport.routes_generator import inductive_route_learning

with initialize_config_dir(config_dir=CFG_DIR, job_name="app"):
    cfg_learn = compose(config_name="ppo_20nodes.yaml", 
                            overrides=["+run_name=test"]) 

inductive_route_learning.setup_and_train(cfg_learn) 

# Experiments pipeline

In [ ]:
import csv
from pathlib import Path
from typing import Dict, Any, List, Tuple

import torch
from torch_geometric.loader import DataLoader
from tqdm import tqdm

# Your custom imports
from transport.routes_generator.citygraph_dataset import get_dataset_from_config
import transport.routes_generator.utils as lrnu
from transport.routes_generator.eval_route_generator import eval_model
from transport.routes_generator.torch_utils import dump_routes


# --------------------------------------------------------------------------- #
#                               Configuration
# --------------------------------------------------------------------------- #
# Path to config directory relative to current notebook
cwd = Path.cwd()

MODEL_WEIGHTS = cwd.parent.parent / "examples/data/model_weights/inductive_random_graphs_weighted_connectivity.pt"

RESULTS_DIR = Path("experiment_results")
RESULTS_DIR.mkdir(exist_ok=True)
UNSERVED_DIR = RESULTS_DIR / "unserved_demand_matrices"
UNSERVED_DIR.mkdir(exist_ok=True)

CSV_HEADER = [
    "dataset", "demand_time", "route_time", "connectivity",
    "ATT", "RTT", "median_connectivity", "median_connectivity_weighted",
    "cost", "$d_{un}$", "$d_0$", "$d_1$", "$d_2$"
]

METRIC_KEYS = [
    'ATT', 'RTT', 'median_connectivity', 'median_connectivity_weighted',
    'cost', '$d_{un}$', '$d_0$', '$d_1$', '$d_2$'
]

DATASETS = ["mandl", "mumford0", "mumford1", "mumford2", "mumford3"]
COEFF_STEPS = [round(i * 0.1, 1) for i in range(11)]
COMBINATIONS = [(dt, round(1.0 - dt, 1), 0.0) for dt in COEFF_STEPS]
EXPERIMENTS = [(ds, dt, rt, ct) for ds in DATASETS for dt, rt, ct in COMBINATIONS]


# --------------------------------------------------------------------------- #
#                               CSV Utils
# --------------------------------------------------------------------------- #
def init_csv(path: Path):
    if not path.exists():
        with open(path, 'w', newline='') as f:
            csv.writer(f).writerow(CSV_HEADER)


def append_row(path: Path, row: List[Any]):
    with open(path, 'a', newline='') as f:
        csv.writer(f).writerow(row)


def save_unserved(matrix: torch.Tensor, prefix: str, tag: str):
    path = UNSERVED_DIR / f"{prefix}_{tag}_unserved_demand.pt"
    torch.save(matrix, path)


# --------------------------------------------------------------------------- #
#                             Core Runners
# --------------------------------------------------------------------------- #
def run_lc_eval(cfg_eval: Any, input_tensors: Any = None):
    """Run LC (neural route construction) evaluation."""
    DEVICE, run_name, _, cost_obj, model = lrnu.process_standard_experiment_cfg(
        cfg_eval, 'nn_construction_', weights_required=True
    )

    test_ds = get_dataset_from_config(cfg_eval.eval.dataset, tensors=input_tensors)
    test_dl = DataLoader(test_ds, batch_size=cfg_eval.batch_size)

    n_samples = cfg_eval.get('n_samples', None)
    sbs = cfg_eval.get('sample_batch_size', cfg_eval.batch_size)

    _, unserved_demand, metrics, routes = eval_model(
        model=model,
        eval_dataloader=test_dl,
        eval_cfg=cfg_eval.eval,
        cost_obj=cost_obj,
        n_samples=n_samples,
        sample_batch_size=sbs,
        return_routes=True,
        silent=True,
        device=DEVICE
    )

    dump_routes(run_name, routes.cpu())
    return metrics, unserved_demand, run_name.split('_', 3)[-1]


def run_bee_optimization(
    cfg_bee: Any,
    input_tensors: Any = None,
    routes_tensor: torch.Tensor | None = None,
    is_neural: bool = True
):

    prefix = 'neural_bco_' if is_neural else 'bco_'
    DEVICE, run_name, sum_writer, cost_obj, bee_model = lrnu.process_standard_experiment_cfg(
        cfg_bee, prefix, weights_required=is_neural
    )

    test_ds = get_dataset_from_config(cfg_bee.eval.dataset, tensors=input_tensors)
    test_dl = DataLoader(test_ds, batch_size=cfg_bee.batch_size)

    force_linking = cfg_bee.get('force_linking_unlinked', False)
    if is_neural and bee_model is not None:
        bee_model.force_linking_unlinked = force_linking
        bee_model.eval()

    nt1b = cfg_bee.get('n_type1_bees', None)
    nt2b = cfg_bee.get('n_type2_bees', None)

    from transport.routes_generator.bee_colony import bee_colony

    test_output = lrnu.test_method(
        bee_colony,
        test_dl,
        cfg_bee.eval,
        cfg_bee.init,
        cost_obj,
        sum_writer=sum_writer,
        silent=True,
        n_bees=cfg_bee.n_bees,
        n_iterations=cfg_bee.n_iterations,
        n_type1_bees=nt1b,
        n_type2_bees=nt2b,
        device=DEVICE,
        bee_model=bee_model if is_neural else None,
        return_routes=True,
        force_linking_unlinked=force_linking,
        routes_tensor=routes_tensor
    )

    routes_out = test_output[-1]
    metrics = test_output[-2]
    unserved_demand = test_output[-3]

    tag = run_name.split('_', 3)[-1] if '_' in run_name else "unknown"
    return metrics, unserved_demand, tag


# --------------------------------------------------------------------------- #
#                               Main Loop
# --------------------------------------------------------------------------- #
lc_csv      = RESULTS_DIR / "lc_results.csv"
neuro_csv   = RESULTS_DIR / "neurobco_results.csv"
bco_csv     = RESULTS_DIR / "bco_results.csv"

for p in (lc_csv, neuro_csv, bco_csv):
    init_csv(p)

for dataset, dt, rt, ct in tqdm(EXPERIMENTS, desc="Experiments"):
    tag = f"{dataset.upper()}_DT{dt:.1f}_RT{rt:.1f}"
    param_tag = f"dt{dt:.1f}_rt{rt:.1f}_ct{ct:.1f}"

    print(f"\n{'='*70}")
    print(f"Processing: {tag}")
    print(f"{'='*70}")

    # ===== Overrides split into NN and non-NN =============================== #
    common_overrides_base = [
        f"+eval={dataset.lower()}",
        f"++experiment.cost_function.kwargs.demand_time_weight={dt}",
        f"++experiment.cost_function.kwargs.route_time_weight={rt}",
        f"++experiment.cost_function.kwargs.median_connectivity_weight={ct}",
    ]

    # LC + NeuroBCO GET weights
    common_overrides_nn = [
        *common_overrides_base,
        f"+model.weights={MODEL_WEIGHTS}",
    ]

    # Classic BCO — NO WEIGHTS
    common_overrides_bco = common_overrides_base[:]

    # ======================== 1. LC ======================================== #
    try:
        with initialize_config_dir(config_dir=str(CFG_DIR), version_base=None):
            cfg_lc = compose(
                config_name="eval_model_mumford",
                overrides=[
                    *common_overrides_nn,
                    f"++run_name=LC_{dataset.upper()}_{param_tag}_init"
                ]
            )
        lc_metrics, lc_unserved, lc_init_tag = run_lc_eval(cfg_lc)
        save_unserved(lc_unserved, "LC", tag)

        row = [dataset.lower(), dt, rt, ct] + \
              [round(lc_metrics.get(k, torch.tensor(0.0)).item(), 4)
               for k in METRIC_KEYS]
        append_row(lc_csv, row)

        print("  [Success] LC completed")

    except Exception as e:
        print(f"  [Error] LC failed: {e}")
        continue

    # Load initial routes for bee methods
    init_routes_path = Path(f"output_routes/nn_construction_LC_{dataset.upper()}_{param_tag}_init_routes.pkl")

    import pickle
    with open(init_routes_path, "rb") as f:
        routes_tensor = pickle.load(f)

    # ======================== 2. NeuroBCO ================================== #
    try:
        with initialize_config_dir(config_dir=str(CFG_DIR), version_base=None):
            cfg_neuro = compose(
                config_name="neural_bco_mumford",
                overrides=[
                    *common_overrides_nn,
                    f"++run_name=NEURO_{dataset.upper()}_{param_tag}_gen",
                    f"init.path={init_routes_path}"
                ]
            )
        neuro_metrics, neuro_unserved, _ = run_bee_optimization(
            cfg_neuro, routes_tensor=routes_tensor, is_neural=True)

        save_unserved(neuro_unserved, "NEURO", tag)

        row = [dataset.lower(), dt, rt, ct] + \
              [round(neuro_metrics.get(k, torch.tensor(0.0)).item(), 4)
               for k in METRIC_KEYS]
        append_row(neuro_csv, row)

        print("  [Success] NeuroBCO completed")

    except Exception as e:
        print(f"  [Error] NeuroBCO failed: {e}")

    # ======================== 3. Classic BCO =============================== #
    try:
        with initialize_config_dir(config_dir=CFG_DIR, version_base=None):
            cfg_bco = compose(
                config_name="bco_mumford",
                overrides=[
                    *common_overrides_bco,
                    f"++run_name=BCO_{dataset.upper()}_{param_tag}_gen",
                    f"init.path={init_routes_path}"
                ]
            )

        bco_metrics, bco_unserved, _ = run_bee_optimization(
            cfg_bco, routes_tensor=routes_tensor, is_neural=False)

        save_unserved(bco_unserved, "BCO", tag)

        row = [dataset.lower(), dt, rt, ct] + \
              [round(bco_metrics.get(k, torch.tensor(0.0)).item(), 4)
               for k in METRIC_KEYS]
        append_row(bco_csv, row)

        print("  [Success] BCO completed")

    except Exception as e:
        print(f"  [Error] BCO failed: {e}")

    print(f"[Success] All methods completed for {tag}")

# --------------------------------------------------------------------------- #
#                                 Final Summary
# --------------------------------------------------------------------------- #
print("\n" + "="*70)
print("=== ALL EXPERIMENTS SUCCESSFULLY COMPLETED ===")
print("="*70)
print(f"Results directory:          {RESULTS_DIR}")
print(f"  - LC results:             {lc_csv}")
print(f"  - NeuroBCO results:       {neuro_csv}")
print(f"  - BCO results:            {bco_csv}")
print(f"  - Unserved demand saved:  {UNSERVED_DIR}/")
